# Chapter 09 답안 양식. 회귀 분석으로 숫자 예측하기

> 이 내용을 `chapter09.ipynb`의 Markdown 셀로 작성합니다.

## 제출 정보
- 이름: 이상재
- GitHub ID: sangjae-lee97
- 작성일: 2026.09.22
- 최종 Notebook URL: https://github.com/sangjae-lee97/kant-axagent-study/blob/main/llm-data-analysis-course/chapter09/chapter09.ipynb

## 1. 예측 문제 정의
- Target: order_total
- 예측 단위: 주문 1건
- 예측 시점: 주문의 기본 정보와 고객 정보가 확보된 시점에서 최종 주문 금액을 예측한다.
- 예측 시점에 알 수 있는 정보:
order_month
order_dayofweek
age
payment_method
gender
city
- 예측 시점에 알 수 없는 정보:
order_total
line_total
quantity
unit_price
item_count
total_quantity
avg_unit_price
order_status
order_id
customer_id
product_id

### 나의 해석과 판단
왜 이 예측 시점을 선택했는지 작성하세요.

주문 기본 정보가 확보된 시점을 바탕으로 최종 주문 금액이 확보되기 전에 " 이 주문은 대략 얼마짜리가 될까?"를 예측하기 위해서

## 2. Feature Leakage Audit

| feature         | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유                                                   |
| --------------- | ----------- | ------------ | -------- | ----- | ------------------------------------------------------- |
| order_month     | O           | X            | X        | O     | 교육용 예측 시점에 사용 가능하다고 가정                                  |
| order_dayofweek | O           | X            | X        | O     | 교육용 예측 시점에 사용 가능하다고 가정                                  |
| age             | O           | X            | X        | O     | 교육용 예측 시점에 사용 가능하다고 가정                                  |
| payment_method  | O           | X            | X        | O     | 교육용 예측 시점에 사용 가능하다고 가정                                  |
| gender          | O           | X            | X        | O     | 교육용 예측 시점에 사용 가능하다고 가정                                  |
| city            | O           | X            | X        | O     | 교육용 예측 시점에 사용 가능하다고 가정                                  |
| avg_unit_price  | X           | O            | O        | X     | 주문 상세에서 계산한 값으로 Target을 간접적으로 알려주는 목표 대리 변수             |
| customer_id     | O           | X            | O        | X     | 예측 시점에 존재할 수 있지만 고객을 구분하는 식별자이므로 제외                     |
| item_count      | X           | X            | O        | X     | 주문 상세가 확인된 뒤 계산되는 사후 집계값                                |
| line_total      | X           | O            | O        | X     | `order_total`을 구성하는 주문 상세 금액이므로 데이터 누수 발생               |
| order_id        | O           | X            | O        | X     | 주문을 구분하기 위한 식별자이며 예측에 의미 있는 일반 특성이 아님                   |
| order_status    | X           | X            | O        | X     | 주문 진행 이후 확정될 수 있는 사후 정보                                 |
| order_total     | X           | X            | O        | X     | 예측하려는 Target 자체이므로 feature로 사용할 수 없음                    |
| product_id      | X           | X            | O        | X     | 주문 상세가 확인되어야 알 수 있는 상품 식별자                              |
| quantity        | X           | O            | O        | X     | `line_total = quantity × unit_price`에 사용되는 Target 계산 재료 |
| total_quantity  | X           | O            | O        | X     | 주문 상세의 quantity를 집계한 값으로 Target을 간접적으로 알려주는 목표 대리 변수    |
| unit_price      | X           | O            | O        | X     | `line_total = quantity × unit_price`에 사용되는 Target 계산 재료 |


- 금지 feature overlap 개수: 0개
- 가장 위험하다고 판단한 feature: order_total
- 이유: 우리가 알아야할 최종 정보이므로 이걸 사용하는 순간 머신러닝이 아닌 이미 아는 정보를 바탕으로 끼워맞추기가 됨

![Leakage 점검](images/step02_leakage.png)

## 3. Target 생성과 관계 검증

### 계산 기준

```text
line_total = quantity * unit_price
order_total = order_id별 line_total 합계
```

In [26]:
import pandas as pd
import numpy as np

# 데이터 불러오기
order_items = pd.read_csv("../data/processed/order_items_clean.csv")
products = pd.read_csv("../data/processed/products_clean.csv")

# 계산값 만들기
expected_line_total = order_items["quantity"] * order_items["unit_price"]

# 기존 line_total과 계산값 비교
is_match = np.isclose(
    order_items["line_total"],
    expected_line_total
)

print("전체 행 수:", len(order_items))
print("일치 행 수:", is_match.sum())
print("불일치 행 수:", (~is_match).sum())

if is_match.all():
    print("line_total 검증 결과: 일치")
else:
    print("line_total 검증 결과: 불일치")

전체 행 수: 764
일치 행 수: 764
불일치 행 수: 0
line_total 검증 결과: 일치


- line_total 검증 결과: 일치
- orders ↔ target 관계 검증 결과: one_to_one
- orders → customers 관계 검증 결과: many_to_one

### 나의 해석과 판단
관계 검증 없이 병합만 수행하면 어떤 문제가 생길 수 있는지 작성하세요.

관계 검증 없이 병합하면 중복 행 증가나 행 누락이 발생할 수 있고, 그 결과 주문 금액과 같은 집계값이 실제보다 크게 또는 작게 계산될 수 있다. 따라서 병합 전에 one_to_one, many_to_one 등의 관계가 예상과 일치하는지 확인해야 한다.


## 4. Train / Final Test 분할

- Train 시작일:2025-07-09
- Train 종료일:2026-04-13
- Train 행 수:244
- Final Test 시작일:2026-04-14
- Final Test 종료일:2026-07-08
- Final Test 행 수:56
- 같은 달력 날짜 중복 여부: 없음

In [27]:
import pandas as pd

# 주문 데이터 불러오기
orders = pd.read_csv("../data/processed/orders_clean.csv")

# 날짜형 변환
orders["order_date"] = pd.to_datetime(orders["order_date"])

# 날짜 기준으로 Train / Test 분리
train = orders[
    (orders["order_date"] >= "2025-07-09") &
    (orders["order_date"] <= "2026-04-13")
].copy()

test = orders[
    (orders["order_date"] >= "2026-04-14") &
    (orders["order_date"] <= "2026-07-08")
].copy()

# 같은 날짜가 양쪽에 존재하는지 확인
overlap_dates = set(train["order_date"].dt.date) & set(test["order_date"].dt.date)

print("Train 행 수:", len(train))
print("Test 행 수:", len(test))
print("같은 달력 날짜 중복 개수:", len(overlap_dates))
print("중복 날짜:", sorted(overlap_dates))

print("Train 마지막 날짜:", train["order_date"].max())
print("Test 첫 날짜:", test["order_date"].min())

print(
    "시간 순서 검증:",
    "PASS" if train["order_date"].max() < test["order_date"].min() else "FAIL"
)

Train 행 수: 244
Test 행 수: 56
같은 달력 날짜 중복 개수: 0
중복 날짜: []
Train 마지막 날짜: 2026-04-13 00:00:00
Test 첫 날짜: 2026-04-14 00:00:00
시간 순서 검증: PASS


### 분할 판단
왜 이번 문제에서 시간 순서 분할을 사용하는지 작성하세요.

우리의 목표는 예측이기 때문에 먼 과거의 데이터를 바탕으로 현재를 예측하는게 목표와 부합하기 때문

![시간 분할](images/step04_split.png)

## 5. Train TimeSeriesSplit 후보 비교

In [28]:
from src.regression import (
    load_regression_source_data,
    build_regression_dataset,
    split_model_data_by_time,
    make_regression_models,
    cross_validate_regression_models,
    select_diagnostic_model,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
)

# 실제 processed 폴더 경로
processed_dir = r"C:\dev\kant-axagent-study\llm-data-analysis-course\data\processed"

# 데이터 불러오기
data = load_regression_source_data(processed_dir)

# 모델링 데이터 생성
model_data = build_regression_dataset(
    customers=data["customers"],
    orders=data["orders"],
    order_items=data["order_items"],
)

# Train / Test 시간 분할
train_data, test_data = split_model_data_by_time(
    model_data,
    test_size=0.2
)

# Train 데이터
X_train = train_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()

# 모델 생성
models = make_regression_models(random_state=42)

# TimeSeriesSplit CV
cv_summary = cross_validate_regression_models(
    models,
    X_train,
    y_train
)

# 비베이스라인 모델 선택
selected_model = select_diagnostic_model(cv_summary)

print(cv_summary)
print()
print("Selected Model:", selected_model)

               model  n_splits    cv_MAE_mean    cv_MAE_std  cv_R2_mean
0      Baseline Mean         5  423004.808743  44153.043466   -0.022821
1      Random Forest         5  437352.803929  26318.793429   -0.109687
2  Linear Regression         5  461386.286905  77792.799324   -0.310517

Selected Model: Random Forest


| 모델                |  CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고        |
| ----------------- | ---------: | ----------: | -------: | --------- |
| Baseline Mean     | 423,004.81 |   44,153.04 |  -0.0228 | 비교 기준     |
| Linear Regression | 461,386.29 |   77,792.80 |  -0.3105 | 비베이스라인 후보 |
| Random Forest     | 437,352.80 |   26,318.79 |  -0.1097 | 선택 모델     |

### Train CV로 선택한 비베이스라인 모델
- Selected Model: Random Forest

- 선택 기준:
비베이스라인 후보인 Linear Regression과 Random Forest 중
CV MAE 평균이 더 낮은 모델을 선택하였다.

- Final Test 결과를 보기 전에 선택했는가: 예

### 나의 해석과 판단
CV 평균뿐 아니라 표준편차도 함께 봐야 하는 이유를 작성하세요.

CV 평균은 모델의 전체적인 성능 수준을 보여주고, 표준편차는 각 검증 구간에서 성능이 얼마나 안정적으로 유지되는지 보여줍니다.

평균이 좋아도 표준편차가 크면 시기에 따라 성능 차이가 많이 날 수 있으므로, 모델의 안정성을 판단하기 위해 둘을 함께 봐야 합니다.

## 6. Final Test: Baseline vs Frozen Model

In [29]:
from src.regression import train_and_evaluate_models

# Final Test 데이터 준비
X_test = test_data[FEATURE_COLUMNS].copy()
y_test = test_data[TARGET_COLUMN].copy()

# Baseline Mean과 이미 선택한 Frozen Model만 최종 평가
model_comparison, predictions = train_and_evaluate_models(
    models=models,
    selected_model_name=selected_model,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)

# 결과 확인
display(model_comparison)

,model,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,baseline,416266.191884,443449.648712,522189.699620,-0.001121,0.00
1,Random Forest,selected_by_train_cv,326438.478132,458819.917234,548190.446723,-0.103298,-3.47



| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Baseline Mean | baseline | 416,266.19 | 443,449.65 | 522,189.70 | -0.0011 | 0.00% |
| Frozen Selected Model (Random Forest) | selected_by_train_cv | 326,438.48 | 458,819.92 | 548,190.45 | -0.1033 | -3.47% |


![최종 회귀 평가](./images/step06_final_metrics.png)

### 결과 관찰

- Baseline Mean의 Test MAE는 약 443,450원이다.
- Random Forest의 Test MAE는 약 458,820원이다.
- Random Forest의 Test RMSE는 약 548,190원이다.
- Random Forest의 Test R²는 -0.1033이다.
- Baseline 대비 MAE 개선율은 -3.47%이다.
- Random Forest의 Train MAE는 약 326,438원이지만 Test MAE는 약 458,820원으로 증가했다.

### 나의 해석과 판단

Random Forest는 Train CV에서 선택된 모델이지만, Final Test에서는 Baseline Mean보다 좋은 성능을 보이지 못했다.

MAE 기준으로 Baseline보다 약 3.47% 성능이 악화되었으며, R²도 음수로 나타나 평균값을 예측하는 단순한 방법보다 Final Test 데이터를 더 잘 설명하지 못했다.

또한 Train MAE보다 Test MAE가 크게 증가했기 때문에 과적합 가능성도 추가로 확인할 필요가 있다.

현재 결과만 보면 Random Forest가 Baseline보다 추가적인 예측 가치를 제공한다고 보기 어렵다.

### 업무·분석적 의미

현재 Random Forest의 Test MAE는 약 458,820원으로, 실제 주문 금액과 예측 금액 사이에 평균적으로 약 46만 원 정도의 차이가 발생한다. 또한 RMSE가 약 548,190원으로 MAE보다 높아 일부 주문에서는 더 큰 예측 오차가 발생할 가능성이 있다.

특히 Baseline Mean보다 MAE가 3.47% 높고 R²도 음수이므로, 현재 모델은 단순히 평균 주문 금액을 예측하는 방법보다 추가적인 예측 가치를 보여주지 못했다. 따라서 현재 수준의 모델을 개별 주문 금액을 정확히 예측해야 하는 업무에 바로 활용하기는 어렵다.

실제 업무에 적용하려면 예측 시점에 사용할 수 있으면서 주문 금액과 관련된 추가 변수를 확보하고, 새로운 미래 데이터에서 다시 성능을 검증할 필요가 있다.

## 7. 대표 오차 사례

In [30]:
from src.regression import create_prediction_result

# 6번에서 평가한 Random Forest의 예측값 가져오기
selected_predictions = predictions[selected_model]

# 실제값, 예측값, residual, absolute error 계산
prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=selected_predictions,
    model_name=selected_model,
)

# 절대오차가 가장 큰 2건 확인
top_errors = (
    prediction_result
    .sort_values("abs_error", ascending=False)
    .head(2)
)

display(
    top_errors[
        [
            "order_date",
            "actual_order_total",
            "predicted_order_total",
            "residual",
            "abs_error",
        ]
    ]
)

,order_date,actual_order_total,predicted_order_total,residual,abs_error
0,2026-06-15,1877000,605528.715709,1.271471e+06,1.271471e+06
1,2026-06-19,1767000,756889.654706,1.010110e+06,1.010110e+06




### 사례 1

- 실제값: 1,877,000원
- 예측값: 약 605,529원
- residual: 약 +1,271,471원
- absolute error: 약 1,271,471원
- 관찰: 실제 주문금액이 예측값보다 크게 높아 약 127만 원을 과소예측했다.
- 가능한 이유 후보: 현재 사용 중인 주문 월, 요일, 나이, 결제수단, 성별, 지역만으로는 이러한 고액 주문의 특성을 충분히 설명하지 못했을 가능성이 있다.
- 추가 확인 사항: 해당 주문이 일반 주문보다 금액이 큰 이유를 설명할 수 있는 예측 시점 이전의 추가 정보가 존재하는지 확인할 필요가 있다.

### 사례 2

- 실제값: 1,767,000원
- 예측값: 약 756,890원
- residual: 약 +1,010,110원
- absolute error: 약 1,010,110원
- 관찰: 실제 주문금액이 예측값보다 높아 약 101만 원을 과소예측했다.
- 가능한 이유 후보: 사례 1과 마찬가지로 현재 feature만으로 고액 주문을 구분할 수 있는 충분한 신호가 부족했을 가능성이 있다.
- 추가 확인 사항: 고액 주문에서 비슷한 과소예측이 반복되는지, 그리고 이를 설명할 수 있는 추가 feature가 있는지 확인할 필요가 있다.

> 원인을 데이터가 직접 보여 주지 않는다면 사실처럼 단정하지 않습니다.

## 8. Validation Evidence

`reports/ch09_regression_validation.csv` 기준으로 작성하세요.

| check | 결과 | 의미 |
|---|---:|---|
| `forbidden_feature_overlap` | 0 / PASS | 금지 feature가 모델 입력에 없음 |
| `strict_train_before_test` | True / PASS | Train이 Final Test보다 시간상 앞에 있음 |
| `selected_model_exists_in_train_cv` | True / PASS | Random Forest가 실제 Train CV 후보였음 |
| `final_test_contains_baseline` | True / PASS | Final Test에 Baseline Mean이 포함됨 |
| `final_test_contains_frozen_selected_model` | True / PASS | CV에서 고정한 Random Forest만 최종 모델로 평가됨 |
| `test_rows_for_r2` | 56 / PASS | R²를 계산하기에 충분한 Test 데이터가 있음 |

### FAIL 항목
- 없음 / 있음: 없음
- 있다면 원인과 수정 내용: 없음

## 9. Evidence와 공개 범위

### 생성된 Evidence
- [x] `ch09_regression_split_summary.csv`
- [x] `ch09_regression_feature_audit.csv`
- [x] `ch09_regression_cv_summary.csv`
- [x] `ch09_regression_model_comparison.csv`
- [x] `ch09_regression_validation.csv`
- [x] `ch09_regression_checklist.csv`

### Internal 파일
- [x] `ch09_regression_model_data_internal.csv`
- [x] `ch09_regression_predictions_internal.csv`

### 공개 결과
- [x] `ch09_regression_report.md`
- [x] `ch09_actual_vs_predicted.png`
- [x] `ch09_residual_histogram.png`

### 공개 범위 판단
왜 `order_id`가 포함된 예측 결과를 Internal로 구분해야 하는지 작성하세요.
order_id는 개별 주문을 식별하기 위한 내부 식별자이므로, 공개 결과에 반드시 필요한 정보는 아닙니다. 또한 예측 결과와 함께 공개될 경우 특정 주문을 추적할 수 있는 가능성이 있어 불필요한 식별 정보 노출 위험이 생길 수 있기때문.


## 10. 최종 사용 판단

- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 데이터로는 사용 보류

### 판단 근거
1. Random Forest의 Test MAE는 약 458,820원으로, Baseline Mean의 약 443,450원보다 높아 Baseline 대비 MAE가 3.47% 악화되었다.
2. Random Forest의 Test R²가 -0.1033으로 나타나 단순 평균 예측보다 Final Test 데이터를 더 잘 설명하지 못했다.
3. Validation 항목은 모두 PASS하여 평가 절차 자체는 적절했지만, 대표 오차 사례에서 약 101만~127만 원의 큰 과소예측이 확인되었다.

### 업무적 위험
현재 모델을 실제 주문 금액 예측에 사용할 경우 평균적으로 약 46만 원의 오차가 발생하며, 일부 주문에서는 100만 원 이상의 큰 오차가 발생할 수 있다. 따라서 예산 산정이나 매출 예측처럼 주문 금액의 정확성이 중요한 업무에 사용하면 잘못된 판단으로 이어질 위험이 있다.

### 현재 데이터의 한계
현재 feature는 주문 월, 요일, 고객 나이, 결제수단, 성별, 지역으로 제한되어 있다. 이러한 정보만으로는 주문 금액, 특히 고액 주문의 차이를 충분히 설명하기 어려울 가능성이 있다.

### 다음 개선 우선순위
1. 예측 시점 이전에 사용할 수 있으면서 주문 금액과 관련된 추가 feature를 탐색한다.
2. 고액 주문에서 큰 과소예측이 반복되는지 잔차와 오차 패턴을 추가 분석한다.
3. 개선 후에는 현재 Final Test를 반복적인 모델 선택에 사용하지 않고 새로운 미래 기간의 데이터로 다시 검증한다.

## 11. 전체 재실행 확인

실행 명령:

```powershell
python scripts/run_regression_analysis.py
```

- 실행 성공 여부: 성공
- Train CV Selected Model: Random Forest
- Final Validation 전체 PASS 여부: 6개 전부 PASS
- Notebook 결과와 스크립트 결과의 일관성: 일치

### 결과 확인

전체 회귀 분석 스크립트를 처음부터 다시 실행한 결과,
Train CV에서 선택된 모델은 Random Forest로 Notebook 결과와 동일했다.

Final Test에서도 Random Forest의 Test MAE는 약 458,820원,
Test RMSE는 약 548,190원, Test R²는 -0.1033,
Baseline 대비 MAE 개선율은 -3.47%로 Notebook에서 확인한 결과와 일치했다.

또한 자동 Validation의 6개 항목이 모두 PASS하였으며,
예측 오차 결과와 Evidence 파일 및 진단 그래프도 정상적으로 생성되었다.

따라서 현재 Chapter 09 분석 결과는 전체 재실행을 통해 재현됨을 확인했다.

### step 16. 현재 예측 요구사항의 한계를 다시 확인합니다

## 현재 예측 요구사항의 한계

### 현재 Target

현재 기본 실습의 Target은 주문 한 건의 총금액인 `order_total`이다.

`order_total`은 같은 `order_id`에 속한 `line_total`의 합계이며,
`line_total`은 `quantity × unit_price`로 계산된다.

### 이 값을 정확히 계산할 수 있는 시점

주문 상세의 `quantity`와 `unit_price`가 모두 확정된 시점에는
`order_total`을 회귀모델로 추정할 필요 없이 정확하게 계산할 수 있다.

### 현재 회귀 예측이 교육용으로는 의미가 있지만 실제 업무 예측으로는 제한적인 이유

현재 실습은 Feature Leakage, 시간 순서 분할, Baseline 비교,
MAE·RMSE·R² 해석, 잔차 분석 등 회귀분석의 전체 절차를 학습하는 데 의미가 있다.

하지만 실제 업무에서는 주문 상세가 확정된 후에는 주문금액을 정확하게 계산할 수 있기 때문에,
같은 값을 머신러닝으로 다시 예측하는 것의 실질적인 활용 가치는 제한적이다.

즉, 회귀모델을 사용할 수 있다는 것과 회귀모델을 사용해야 할 업무적 이유가 있다는 것은 다르다.

### 더 의미 있는 예측 문제라면 무엇이 달라져야 하는가

더 의미 있는 회귀 문제는 예측 시점에는 아직 알 수 없는 미래의 연속형 값을 Target으로 해야 한다.

예를 들어 과거 판매·구매 데이터를 이용하여 다음 달 카테고리별 매출,
다음 기간 상품별 판매 수량 또는 고객의 향후 구매금액을 예측할 수 있다.

이 경우 예측 결과를 재고, 판매, 마케팅, 예산 등의 실제 의사결정에 활용할 수 있어
현재 `order_total` 예측보다 업무적 의미가 크다.

### step 17. 실제로 의미 있는 새로운 회귀 예측 요구사항을 정의합니다

In [31]:
# STEP 17 후보 예측 문제를 고르기 위한 데이터 확인

import pandas as pd

# 날짜형 확인
orders["order_date"] = pd.to_datetime(orders["order_date"])

print("=== 전체 데이터 기간 ===")
print("시작일:", orders["order_date"].min())
print("종료일:", orders["order_date"].max())
print("개월 수:", orders["order_date"].dt.to_period("M").nunique())
print("주 수:", orders["order_date"].dt.to_period("W").nunique())


# 1. 고객별 향후 구매금액 예측 가능성
customer_order_counts = (
    orders.groupby("customer_id")
    .size()
    .sort_values(ascending=False)
)

print("\n=== 고객 반복 구매 현황 ===")
print("고객 수:", customer_order_counts.size)
print("고객당 평균 주문 수:", round(customer_order_counts.mean(), 2))
print("2회 이상 주문 고객:", (customer_order_counts >= 2).sum())
print("3회 이상 주문 고객:", (customer_order_counts >= 3).sum())
print("5회 이상 주문 고객:", (customer_order_counts >= 5).sum())
print("\n고객별 주문 수 상위:")
display(customer_order_counts.head(10).to_frame("order_count"))


# 2. 상품별 판매량 예측 가능성
print("\n=== 상품 관련 컬럼 확인 ===")
print("order_items 컬럼:", order_items.columns.tolist())

if "product_id" in order_items.columns:
    print("상품 수:", order_items["product_id"].nunique())
    print(
        "상품당 평균 주문상세 수:",
        round(order_items.groupby("product_id").size().mean(), 2)
    )
else:
    print("product_id가 없어 상품별 예측은 바로 만들기 어렵습니다.")


# 3. 카테고리별 매출 예측 가능성
print("\n=== 카테고리 관련 컬럼 확인 ===")

category_columns = [
    col
    for col in order_items.columns
    if "category" in col.lower()
]

print("order_items의 category 관련 컬럼:", category_columns)

print("\n=== 판단에 사용할 핵심 질문 ===")
print("1. 고객별 반복 구매가 충분한가?")
print("2. product_id가 존재하는가?")
print("3. category 정보를 현재 데이터에서 연결할 수 있는가?")
print("4. 월 단위 예측을 만들 만큼 기간이 충분한가?")

=== 전체 데이터 기간 ===
시작일: 2025-07-09 00:00:00
종료일: 2026-07-08 00:00:00
개월 수: 13
주 수: 53

=== 고객 반복 구매 현황 ===
고객 수: 126
고객당 평균 주문 수: 2.38
2회 이상 주문 고객: 89
3회 이상 주문 고객: 51
5회 이상 주문 고객: 8

고객별 주문 수 상위:


,order_count
customer_id,
30,7
24,6
117,6
40,5
83,5
102,5
98,5
114,5
105,4



=== 상품 관련 컬럼 확인 ===
order_items 컬럼: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']
상품 수: 100
상품당 평균 주문상세 수: 7.64

=== 카테고리 관련 컬럼 확인 ===
order_items의 category 관련 컬럼: []

=== 판단에 사용할 핵심 질문 ===
1. 고객별 반복 구매가 충분한가?
2. product_id가 존재하는가?
3. category 정보를 현재 데이터에서 연결할 수 있는가?
4. 월 단위 예측을 만들 만큼 기간이 충분한가?


| 후보 | 현재 판단 | 이유 |
|---|---|---|
| 고객별 향후 구매금액 | 가능하지만 다소 희소 | 고객당 평균 주문 2.38건 |
| 상품별 다음 기간 판매량 | 권장도 낮음 | 상품 100개, 상품당 평균 상세 7.64건 |
| 카테고리별 다음 달 매출 | **가장 유망** | 상품을 집계해 샘플 희소성을 줄일 수 있음 |

In [32]:
# products 데이터 확인

print("products 컬럼:")
print(products.columns.tolist())

if "category" in products.columns:
    print("\n카테고리 수:", products["category"].nunique())
    print("\n카테고리별 상품 수:")
    display(
        products["category"]
        .value_counts()
        .to_frame("product_count")
    )
else:
    print("\nproducts에도 category 컬럼이 없습니다.")

products 컬럼:
['product_id', 'product_name', 'category', 'price']

카테고리 수: 7

카테고리별 상품 수:


,product_count
category,
스포츠,19
전자기기,17
생활용품,16
뷰티,16
도서,14
패션,11
식품,7


## 실제로 의미 있는 새로운 회귀 예측 요구사항

### 선택한 예측 문제

과거 판매 데이터를 이용하여 다음 달 카테고리별 completed 주문 매출을 예측한다.

### 선택 이유

현재 데이터에는 100개의 상품과 7개의 카테고리가 존재한다.
개별 상품별로 예측할 경우 상품당 판매 기록이 많지 않아 데이터가 지나치게 희소해질 가능성이 있다.

반면 카테고리 단위로 여러 상품의 판매를 집계하면 개별 상품보다 안정적인 판매 이력을 만들 수 있으며,
다음 달 매출은 현재 시점에서는 정확하게 알 수 없는 미래의 연속형 값이므로 회귀 예측의 목적에도 적합하다.

또한 다음 달 카테고리별 예상 매출은 재고 계획, 판매 계획, 프로모션 검토,
예산 계획 등의 의사결정에 활용할 수 있다.

### 다른 후보를 선택하지 않은 이유

- 고객별 향후 구매금액:
  고객당 평균 주문 수가 약 2.38건이고 5회 이상 구매한 고객이 8명뿐이어서
  고객별 반복 구매 패턴을 학습하기에는 데이터가 다소 희소하다.

- 상품별 다음 기간 판매 수량:
  상품이 100개이고 상품당 평균 주문상세 수가 약 7.64건이므로,
  월 단위 상품별 데이터를 만들 경우 판매 기록이 없는 구간이 많아질 가능성이 있다.

따라서 현재 데이터에서는 상품이나 고객 단위보다 카테고리 단위로 집계한
다음 달 매출 예측이 상대적으로 적합하다고 판단했다.

### 현재 데이터의 한계

전체 데이터 기간이 13개월로 짧기 때문에 카테고리 단위로 집계하더라도
학습 가능한 시계열 표본 수가 많지 않을 수 있다.

따라서 이후 단계에서 실제 category × month 데이터의 행 수와
월별 판매 기록의 연속성을 확인한 뒤 모델링 가능성을 다시 판단한다.

### STEP 18. 최종 과제의 Prediction Contract를 작성합니다

## 최종 과제 Prediction Contract

### 예측 목적

과거 카테고리별 판매 데이터를 이용하여 다음 달 카테고리별 매출을 미리 예측한다.

예측 결과는 다음 달의 재고 계획, 판매 계획, 프로모션 검토,
예산 계획 등의 의사결정을 지원하는 데 활용한다.

### Target

다음 달 카테고리별 completed 주문의 총매출이다.

카테고리별 월 매출은 해당 월에 완료된 주문의 `line_total`을
카테고리별로 합산하여 계산한다.

`line_total = quantity × unit_price`

### 예측 단위

`category × month`

즉 한 행은 특정 카테고리의 특정 월을 의미한다.

예:
- 스포츠 × 2026-05
- 전자기기 × 2026-05
- 식품 × 2026-05

### 예측 시점

매월 말일을 기준으로 예측한다.

예를 들어 2026년 5월 말까지 확인된 판매 데이터를 이용하여
2026년 6월의 카테고리별 매출을 예측한다.

### Prediction Horizon

다음 1개월

### 사용할 Feature

예측 시점까지 이미 발생한 과거 판매 정보만 사용한다.

후보 Feature는 다음과 같다.

- category
- 예측 대상 월 정보
- 직전 1개월 매출
- 최근 3개월 평균 매출
- 직전 1개월 주문 건수
- 직전 1개월 판매 수량
- 최근 3개월 평균 주문 건수
- 최근 3개월 평균 판매 수량

모든 과거 집계 Feature는 예측하려는 달보다 이전 데이터만 사용하여 생성한다.

### 사용하지 않을 Feature와 이유

다음 달의 실제 주문 정보는 예측 시점에는 알 수 없으므로 사용하지 않는다.

예:

- 다음 달 실제 매출
- 다음 달 주문 건수
- 다음 달 판매 수량
- 다음 달 quantity
- 다음 달 unit_price
- 다음 달 line_total
- 다음 달 주문 정보에서 계산한 모든 집계값

이러한 값을 사용하면 미래 정보가 Feature에 포함되어 데이터 누수가 발생한다.

### 예측 결과의 실제 활용 방법

다음 달 카테고리별 예상 매출을 이용하여 다음과 같은 의사결정을 지원할 수 있다.

- 매출이 증가할 것으로 예상되는 카테고리의 재고 확보 검토
- 판매가 감소할 것으로 예상되는 카테고리의 프로모션 검토
- 카테고리별 판매 계획 수립
- 다음 달 매출 및 예산 계획 수립

### STEP 19. 새로운 Target과 Feature 데이터셋을 만듭니다

In [34]:
import pandas as pd
from pathlib import Path

# ==================================================
# STEP 19. 새로운 Target과 Feature 데이터셋 만들기
# - 다음 달 카테고리별 매출 예측
# - 완전한 월만 사용
# ==================================================


# --------------------------------------------------
# 1. 전처리된 데이터 불러오기
# --------------------------------------------------
processed_dir = Path("../data/processed")

orders = pd.read_csv(
    processed_dir / "orders_clean.csv"
)

order_items = pd.read_csv(
    processed_dir / "order_items_clean.csv"
)

products = pd.read_csv(
    processed_dir / "products_clean.csv"
)

# 날짜형 변환
orders["order_date"] = pd.to_datetime(
    orders["order_date"]
)


# --------------------------------------------------
# 2. 원본 데이터 기간 확인
# --------------------------------------------------
first_date = orders["order_date"].min()
last_date = orders["order_date"].max()

print("=== 원본 데이터 기간 ===")
print("시작일:", first_date)
print("종료일:", last_date)


# --------------------------------------------------
# 3. 양 끝의 불완전한 월 제외
# --------------------------------------------------
# 시작일이 2025-07-09이므로 2025-07은 부분 월
# 종료일이 2026-07-08이므로 2026-07도 부분 월
#
# 따라서 완전한 월:
# 2025-08 ~ 2026-06

first_full_month = (
    first_date.to_period("M") + 1
)

last_full_month = (
    last_date.to_period("M") - 1
)

all_months = pd.period_range(
    start=first_full_month,
    end=last_full_month,
    freq="M",
)

print("\n=== 사용할 완전한 월 ===")
print("시작 월:", all_months.min())
print("마지막 월:", all_months.max())
print("개월 수:", len(all_months))


# --------------------------------------------------
# 4. completed 주문만 선택
# --------------------------------------------------
completed_orders = orders[
    orders["order_status"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("completed")
].copy()


# --------------------------------------------------
# 5. 주문 + 주문상세 + 상품 카테고리 결합
# --------------------------------------------------
sales_data = (
    completed_orders[
        [
            "order_id",
            "order_date",
        ]
    ]
    .merge(
        order_items[
            [
                "order_id",
                "product_id",
                "quantity",
                "unit_price",
            ]
        ],
        on="order_id",
        how="inner",
        validate="one_to_many",
    )
    .merge(
        products[
            [
                "product_id",
                "category",
            ]
        ],
        on="product_id",
        how="left",
        validate="many_to_one",
    )
)


# --------------------------------------------------
# 6. 매출 계산
# --------------------------------------------------
sales_data["sales"] = (
    sales_data["quantity"]
    * sales_data["unit_price"]
)

# 월 단위 컬럼 생성
sales_data["month"] = (
    sales_data["order_date"]
    .dt.to_period("M")
)


# --------------------------------------------------
# 7. 완전한 월 데이터만 사용
# --------------------------------------------------
sales_data = sales_data[
    sales_data["month"].isin(all_months)
].copy()


# --------------------------------------------------
# 8. category × month 단위 집계
# --------------------------------------------------
monthly_category = (
    sales_data
    .groupby(
        [
            "category",
            "month",
        ],
        as_index=False,
    )
    .agg(
        total_sales=(
            "sales",
            "sum",
        ),
        order_count=(
            "order_id",
            "nunique",
        ),
        total_quantity=(
            "quantity",
            "sum",
        ),
    )
)


# --------------------------------------------------
# 9. 판매가 없었던 달도 0으로 포함
# --------------------------------------------------
all_categories = sorted(
    products["category"]
    .dropna()
    .unique()
)

full_index = pd.MultiIndex.from_product(
    [
        all_categories,
        all_months,
    ],
    names=[
        "category",
        "month",
    ],
)

monthly_category = (
    monthly_category
    .set_index(
        [
            "category",
            "month",
        ]
    )
    .reindex(
        full_index,
        fill_value=0,
    )
    .reset_index()
    .sort_values(
        [
            "category",
            "month",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------
# 10. 카테고리별 그룹 생성
# --------------------------------------------------
grouped = monthly_category.groupby(
    "category"
)


# --------------------------------------------------
# 11. 직전 1개월 Feature
# --------------------------------------------------
# 현재 행의 month는 예측 시점에서 "직전 월"
monthly_category["last_month_sales"] = (
    monthly_category["total_sales"]
)

monthly_category["last_month_orders"] = (
    monthly_category["order_count"]
)

monthly_category["last_month_quantity"] = (
    monthly_category["total_quantity"]
)


# --------------------------------------------------
# 12. 최근 3개월 평균 Feature
# --------------------------------------------------
monthly_category["recent_3m_avg_sales"] = (
    grouped["total_sales"]
    .transform(
        lambda s: s.rolling(
            window=3,
            min_periods=3,
        ).mean()
    )
)

monthly_category["recent_3m_avg_orders"] = (
    grouped["order_count"]
    .transform(
        lambda s: s.rolling(
            window=3,
            min_periods=3,
        ).mean()
    )
)

monthly_category["recent_3m_avg_quantity"] = (
    grouped["total_quantity"]
    .transform(
        lambda s: s.rolling(
            window=3,
            min_periods=3,
        ).mean()
    )
)


# --------------------------------------------------
# 13. Target = 다음 달 카테고리 매출
# --------------------------------------------------
monthly_category["target_next_month_sales"] = (
    grouped["total_sales"]
    .shift(-1)
)

# 예측하려는 다음 달
monthly_category["target_month"] = (
    monthly_category["month"] + 1
)


# --------------------------------------------------
# 14. 예측 대상 월 정보 Feature
# --------------------------------------------------
# 예: 2026-06을 예측한다면 month_of_year = 6
monthly_category["target_month_of_year"] = (
    monthly_category["target_month"]
    .dt.month
)


# --------------------------------------------------
# 15. 모델링 가능한 행만 남기기
# --------------------------------------------------
# 최근 3개월 데이터가 있어야 하고
# 다음 달 실제 매출도 존재해야 함
feature_dataset = (
    monthly_category
    .dropna(
        subset=[
            "recent_3m_avg_sales",
            "recent_3m_avg_orders",
            "recent_3m_avg_quantity",
            "target_next_month_sales",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------
# 16. 결과 확인
# --------------------------------------------------
print("\n=== 원본 processed 데이터 ===")
print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("products:", products.shape)

print("\n=== category × month 데이터 ===")
print(
    "카테고리 수:",
    monthly_category["category"].nunique()
)
print(
    "개월 수:",
    monthly_category["month"].nunique()
)
print(
    "전체 행 수:",
    len(monthly_category)
)

print("\n=== 최종 모델링 데이터 ===")
print(
    "행 수:",
    len(feature_dataset)
)
print(
    "예측 대상 기간:",
    feature_dataset["target_month"].min(),
    "~",
    feature_dataset["target_month"].max(),
)

print("\n=== 기대값 확인 ===")
print(
    "category × month 원본:",
    len(all_categories),
    "×",
    len(all_months),
    "=",
    len(all_categories) * len(all_months)
)

print(
    "모델링 가능한 예측 월 수:",
    feature_dataset["target_month"].nunique()
)


# --------------------------------------------------
# 17. 최종 데이터 미리보기
# --------------------------------------------------
display(
    feature_dataset[
        [
            "category",
            "month",
            "target_month",
            "target_month_of_year",
            "last_month_sales",
            "recent_3m_avg_sales",
            "last_month_orders",
            "recent_3m_avg_orders",
            "last_month_quantity",
            "recent_3m_avg_quantity",
            "target_next_month_sales",
        ]
    ].head(15)
)

=== 원본 데이터 기간 ===
시작일: 2025-07-09 00:00:00
종료일: 2026-07-08 00:00:00

=== 사용할 완전한 월 ===
시작 월: 2025-08
마지막 월: 2026-06
개월 수: 11

=== 원본 processed 데이터 ===
orders: (300, 7)
order_items: (764, 6)
products: (100, 4)

=== category × month 데이터 ===
카테고리 수: 7
개월 수: 11
전체 행 수: 77

=== 최종 모델링 데이터 ===
행 수: 56
예측 대상 기간: 2025-11 ~ 2026-06

=== 기대값 확인 ===
category × month 원본: 7 × 11 = 77
모델링 가능한 예측 월 수: 8


,category,month,target_month,target_month_of_year,last_month_sales,recent_3m_avg_sales,last_month_orders,recent_3m_avg_orders,last_month_quantity,recent_3m_avg_quantity,target_next_month_sales
0,도서,2025-10,2025-11,11,2550000,1.307333e+06,6,4.000000,19,11.000000,870000.0
1,도서,2025-11,2025-12,12,870000,1.421333e+06,1,3.333333,5,10.666667,1975000.0
2,도서,2025-12,2026-01,1,1975000,1.798333e+06,6,4.333333,18,14.000000,2352000.0
3,도서,2026-01,2026-02,2,2352000,1.732333e+06,7,4.666667,18,13.666667,1044000.0
4,도서,2026-02,2026-03,3,1044000,1.790333e+06,5,6.000000,10,15.333333,990000.0
5,도서,2026-03,2026-04,4,990000,1.462000e+06,4,5.333333,12,13.333333,2925000.0
6,도서,2026-04,2026-05,5,2925000,1.653000e+06,10,6.333333,28,16.666667,1313000.0
7,도서,2026-05,2026-06,6,1313000,1.742667e+06,3,5.666667,15,18.333333,0.0
8,뷰티,2025-10,2025-11,11,4606000,2.913333e+06,9,7.333333,39,24.333333,1317000.0
9,뷰티,2025-11,2025-12,12,1317000,2.498667e+06,6,7.000000,17,24.333333,2305000.0


### 데이터 기간 처리

원본 데이터는 2025-07-09부터 2026-07-08까지 존재하므로
2025년 7월과 2026년 7월은 완전한 월 데이터가 아니다.

이번 과제의 Target은 월 전체 매출이므로 부분 월을 포함하면 월별 매출이
실제보다 작게 계산될 수 있다.

따라서 양 끝의 부분 월을 제외하고
2025-08부터 2026-06까지의 완전한 월만 사용하였다.

In [35]:
# ==================================================
# STEP 19-2. Feature Leakage Audit
# ==================================================

import pandas as pd

feature_audit = pd.DataFrame([
    {
        "Feature": "category",
        "예측 시점 사용 가능": "가능",
        "미래 정보 포함": "아니오",
        "최종 사용": "사용",
        "이유": "예측하려는 카테고리는 예측 시점에 이미 정해져 있음",
    },
    {
        "Feature": "target_month_of_year",
        "예측 시점 사용 가능": "가능",
        "미래 정보 포함": "아니오",
        "최종 사용": "사용",
        "이유": "어느 달을 예측하는지는 예측 시점에 알고 있는 달력 정보임",
    },
    {
        "Feature": "last_month_sales",
        "예측 시점 사용 가능": "가능",
        "미래 정보 포함": "아니오",
        "최종 사용": "사용",
        "이유": "직전 월까지 확정된 카테고리 매출",
    },
    {
        "Feature": "recent_3m_avg_sales",
        "예측 시점 사용 가능": "가능",
        "미래 정보 포함": "아니오",
        "최종 사용": "사용",
        "이유": "예측 대상 월 이전 최근 3개월 매출만 사용",
    },
    {
        "Feature": "last_month_orders",
        "예측 시점 사용 가능": "가능",
        "미래 정보 포함": "아니오",
        "최종 사용": "사용",
        "이유": "직전 월까지 확정된 주문 건수",
    },
    {
        "Feature": "recent_3m_avg_orders",
        "예측 시점 사용 가능": "가능",
        "미래 정보 포함": "아니오",
        "최종 사용": "사용",
        "이유": "예측 대상 월 이전 최근 3개월 주문 건수 평균",
    },
    {
        "Feature": "last_month_quantity",
        "예측 시점 사용 가능": "가능",
        "미래 정보 포함": "아니오",
        "최종 사용": "사용",
        "이유": "직전 월까지 확정된 판매 수량",
    },
    {
        "Feature": "recent_3m_avg_quantity",
        "예측 시점 사용 가능": "가능",
        "미래 정보 포함": "아니오",
        "최종 사용": "사용",
        "이유": "예측 대상 월 이전 최근 3개월 판매 수량 평균",
    },
    {
        "Feature": "month",
        "예측 시점 사용 가능": "가능",
        "미래 정보 포함": "아니오",
        "최종 사용": "미사용",
        "이유": "과거 기준 월 확인용이며 모델 입력에는 별도로 사용하지 않음",
    },
    {
        "Feature": "target_month",
        "예측 시점 사용 가능": "가능",
        "미래 정보 포함": "아니오",
        "최종 사용": "미사용",
        "이유": "시간 순서 분할과 결과 확인에만 사용",
    },
    {
        "Feature": "total_sales",
        "예측 시점 사용 가능": "가능",
        "미래 정보 포함": "아니오",
        "최종 사용": "미사용",
        "이유": "last_month_sales와 동일한 정보이므로 중복 사용하지 않음",
    },
    {
        "Feature": "order_count",
        "예측 시점 사용 가능": "가능",
        "미래 정보 포함": "아니오",
        "최종 사용": "미사용",
        "이유": "last_month_orders와 동일한 정보이므로 중복 사용하지 않음",
    },
    {
        "Feature": "total_quantity",
        "예측 시점 사용 가능": "가능",
        "미래 정보 포함": "아니오",
        "최종 사용": "미사용",
        "이유": "last_month_quantity와 동일한 정보이므로 중복 사용하지 않음",
    },
    {
        "Feature": "target_next_month_sales",
        "예측 시점 사용 가능": "불가능",
        "미래 정보 포함": "예",
        "최종 사용": "Target",
        "이유": "예측하려는 다음 달 실제 매출이므로 Feature로 사용하면 Target Leakage",
    },
])

display(feature_audit)


# 실제 모델에 사용할 Feature 목록
FEATURE_COLUMNS = [
    "category",
    "target_month_of_year",
    "last_month_sales",
    "recent_3m_avg_sales",
    "last_month_orders",
    "recent_3m_avg_orders",
    "last_month_quantity",
    "recent_3m_avg_quantity",
]

TARGET_COLUMN = "target_next_month_sales"

print("사용 Feature:")
print(FEATURE_COLUMNS)

print("\nTarget:")
print(TARGET_COLUMN)

,Feature,예측 시점 사용 가능,미래 정보 포함,최종 사용,이유
0,category,가능,아니오,사용,예측하려는 카테고리는 예측 시점에 이미 정해져 있음
1,target_month_of_year,가능,아니오,사용,어느 달을 예측하는지는 예측 시점에 알고 있는 달력 정보임
2,last_month_sales,가능,아니오,사용,직전 월까지 확정된 카테고리 매출
3,recent_3m_avg_sales,가능,아니오,사용,예측 대상 월 이전 최근 3개월 매출만 사용
4,last_month_orders,가능,아니오,사용,직전 월까지 확정된 주문 건수
5,recent_3m_avg_orders,가능,아니오,사용,예측 대상 월 이전 최근 3개월 주문 건수 평균
6,last_month_quantity,가능,아니오,사용,직전 월까지 확정된 판매 수량
7,recent_3m_avg_quantity,가능,아니오,사용,예측 대상 월 이전 최근 3개월 판매 수량 평균
8,month,가능,아니오,미사용,과거 기준 월 확인용이며 모델 입력에는 별도로 사용하지 않음
9,target_month,가능,아니오,미사용,시간 순서 분할과 결과 확인에만 사용


사용 Feature:
['category', 'target_month_of_year', 'last_month_sales', 'recent_3m_avg_sales', 'last_month_orders', 'recent_3m_avg_orders', 'last_month_quantity', 'recent_3m_avg_quantity']

Target:
target_next_month_sales


### STEP 20. 시간 순서로 학습·평가하고 Baseline과 비교합니다

Train
2025-11 ~ 2026-04
→ 6개월 × 7카테고리 = 42행

Final Test
2026-05 ~ 2026-06
→ 2개월 × 7카테고리 = 14행

In [36]:
# ==================================================
# STEP 20-1. 시간 순서 Train / Final Test 분리
# ==================================================

# 예측 대상 월을 시간순으로 정렬
target_months = sorted(
    feature_dataset["target_month"].unique()
)

# 전체 기간의 약 80%를 Train으로 사용
split_idx = int(len(target_months) * 0.8)

train_months = target_months[:split_idx]
test_months = target_months[split_idx:]


# --------------------------------------------------
# Train / Final Test 분리
# --------------------------------------------------
train_data = feature_dataset[
    feature_dataset["target_month"].isin(train_months)
].copy()

test_data = feature_dataset[
    feature_dataset["target_month"].isin(test_months)
].copy()


# --------------------------------------------------
# Feature / Target 분리
# --------------------------------------------------
X_train = train_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()

X_test = test_data[FEATURE_COLUMNS].copy()
y_test = test_data[TARGET_COLUMN].copy()


# --------------------------------------------------
# 결과 확인
# --------------------------------------------------
print("=== 시간 순서 분할 ===")

print("\n전체 예측 월:")
print(target_months)

print("\nTrain 예측 월:")
print(train_months)

print("\nFinal Test 예측 월:")
print(test_months)

print("\nTrain 행 수:", len(train_data))
print("Final Test 행 수:", len(test_data))

print(
    "\nTrain 마지막 Target 월:",
    train_data["target_month"].max()
)

print(
    "Test 첫 Target 월:",
    test_data["target_month"].min()
)


# --------------------------------------------------
# 시간 순서 검증
# --------------------------------------------------
time_order_pass = (
    train_data["target_month"].max()
    <
    test_data["target_month"].min()
)

print(
    "\n시간 순서 검증:",
    "PASS" if time_order_pass else "FAIL"
)


# 같은 Target 월이 양쪽에 들어갔는지 확인
overlap_months = (
    set(train_data["target_month"])
    &
    set(test_data["target_month"])
)

print(
    "Train/Test 중복 Target 월:",
    overlap_months
)

=== 시간 순서 분할 ===

전체 예측 월:
[Period('2025-11', 'M'), Period('2025-12', 'M'), Period('2026-01', 'M'), Period('2026-02', 'M'), Period('2026-03', 'M'), Period('2026-04', 'M'), Period('2026-05', 'M'), Period('2026-06', 'M')]

Train 예측 월:
[Period('2025-11', 'M'), Period('2025-12', 'M'), Period('2026-01', 'M'), Period('2026-02', 'M'), Period('2026-03', 'M'), Period('2026-04', 'M')]

Final Test 예측 월:
[Period('2026-05', 'M'), Period('2026-06', 'M')]

Train 행 수: 42
Final Test 행 수: 14

Train 마지막 Target 월: 2026-04
Test 첫 Target 월: 2026-05

시간 순서 검증: PASS
Train/Test 중복 Target 월: set()


### STEP 20-2. Baseline과 후보 모델을 Train 안에서 비교

이번 Baseline은 이전 실습의 평균 예측보다 지금 문제에 더 잘 맞는 방식으로 하자.
다음 달 매출 = 직전 달 매출

예를 들어 스포츠 3월 매출이 200만 원이었다면, 아무 모델 없이 스포츠 4월도 200만 원이라고 예측하는 거야.
이 Baseline을 Linear Regression, Random Forest와 비교하자.

일반적인 행 단위 TimeSeriesSplit 대신 월 단위 expanding validation을 직접 만들 거야.
Fold 1
Train: 2025-11 ~ 2025-12
Valid: 2026-01

Fold 2
Train: 2025-11 ~ 2026-01
Valid: 2026-02

Fold 3
Train: 2025-11 ~ 2026-02
Valid: 2026-03

Fold 4
Train: 2025-11 ~ 2026-03
Valid: 2026-04

In [37]:
# ==================================================
# STEP 20-2. Train 내부 시간순서 검증
# Baseline vs Linear Regression vs Random Forest
# ==================================================

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# --------------------------------------------------
# 1. 사용할 Feature 구분
# --------------------------------------------------
categorical_features = [
    "category"
]

numeric_features = [
    "target_month_of_year",
    "last_month_sales",
    "recent_3m_avg_sales",
    "last_month_orders",
    "recent_3m_avg_orders",
    "last_month_quantity",
    "recent_3m_avg_quantity",
]


# --------------------------------------------------
# 2. 전처리기
# --------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features,
        ),
    ]
)


# --------------------------------------------------
# 3. 후보 모델
# --------------------------------------------------
models = {
    "Linear Regression": Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "model",
                LinearRegression(),
            ),
        ]
    ),

    "Random Forest": Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=200,
                    random_state=42,
                ),
            ),
        ]
    ),
}


# --------------------------------------------------
# 4. Train 내부의 예측 대상 월
# --------------------------------------------------
cv_months = sorted(
    train_data["target_month"].unique()
)

print("=== Train 내부 예측 월 ===")
print(cv_months)


# --------------------------------------------------
# 5. 월 단위 Expanding Validation
#
# 처음 2개월은 학습에 사용하고,
# 그 다음 달부터 한 달씩 검증
# --------------------------------------------------
cv_results = []

for valid_idx in range(2, len(cv_months)):

    valid_month = cv_months[valid_idx]

    fold_train_months = cv_months[:valid_idx]

    fold_train = train_data[
        train_data["target_month"].isin(
            fold_train_months
        )
    ].copy()

    fold_valid = train_data[
        train_data["target_month"]
        .eq(valid_month)
    ].copy()


    X_fold_train = fold_train[
        FEATURE_COLUMNS
    ].copy()

    y_fold_train = fold_train[
        TARGET_COLUMN
    ].copy()

    X_fold_valid = fold_valid[
        FEATURE_COLUMNS
    ].copy()

    y_fold_valid = fold_valid[
        TARGET_COLUMN
    ].copy()


    # ----------------------------------------------
    # Baseline
    # 다음 달 매출 = 직전 달 매출
    # ----------------------------------------------
    baseline_pred = (
        fold_valid["last_month_sales"]
        .to_numpy()
    )

    cv_results.append({
        "fold": valid_idx - 1,
        "validation_month": str(valid_month),
        "model": "Baseline - Previous Month",
        "MAE": mean_absolute_error(
            y_fold_valid,
            baseline_pred,
        ),
        "RMSE": mean_squared_error(
            y_fold_valid,
            baseline_pred,
        ) ** 0.5,
        "R2": r2_score(
            y_fold_valid,
            baseline_pred,
        ),
    })


    # ----------------------------------------------
    # 회귀 모델
    # ----------------------------------------------
    for model_name, model in models.items():

        model.fit(
            X_fold_train,
            y_fold_train,
        )

        pred = model.predict(
            X_fold_valid
        )

        cv_results.append({
            "fold": valid_idx - 1,
            "validation_month": str(valid_month),
            "model": model_name,
            "MAE": mean_absolute_error(
                y_fold_valid,
                pred,
            ),
            "RMSE": mean_squared_error(
                y_fold_valid,
                pred,
            ) ** 0.5,
            "R2": r2_score(
                y_fold_valid,
                pred,
            ),
        })


# --------------------------------------------------
# 6. Fold별 결과
# --------------------------------------------------
cv_results = pd.DataFrame(
    cv_results
)

print("\n=== Fold별 결과 ===")
display(cv_results)


# --------------------------------------------------
# 7. 모델별 평균 성능
# --------------------------------------------------
cv_summary = (
    cv_results
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        cv_MAE_mean=("MAE", "mean"),
        cv_MAE_std=("MAE", "std"),
        cv_RMSE_mean=("RMSE", "mean"),
        cv_R2_mean=("R2", "mean"),
    )
    .sort_values(
        "cv_MAE_mean"
    )
    .reset_index(drop=True)
)

print("\n=== Train CV 모델 비교 ===")
display(cv_summary)

=== Train 내부 예측 월 ===
[Period('2025-11', 'M'), Period('2025-12', 'M'), Period('2026-01', 'M'), Period('2026-02', 'M'), Period('2026-03', 'M'), Period('2026-04', 'M')]

=== Fold별 결과 ===


,fold,validation_month,model,MAE,RMSE,R2
0,1,2026-01,Baseline - Previous Month,1.300571e+06,1.984863e+06,-0.821151
1,1,2026-01,Linear Regression,1.130899e+06,1.288199e+06,0.232902
2,1,2026-01,Random Forest,1.109479e+06,1.802504e+06,-0.501888
3,2,2026-02,Baseline - Previous Month,1.204286e+06,1.579888e+06,-4.782200
4,2,2026-02,Linear Regression,3.508364e+06,4.466146e+06,-45.206880
5,2,2026-02,Random Forest,7.892886e+05,9.499098e+05,-1.090285
6,3,2026-03,Baseline - Previous Month,6.580000e+05,7.514149e+05,0.286177
7,3,2026-03,Linear Regression,7.394794e+05,8.835503e+05,0.013053
8,3,2026-03,Random Forest,6.071314e+05,7.838864e+05,0.223150
9,4,2026-04,Baseline - Previous Month,1.484857e+06,1.604515e+06,-1.537623



=== Train CV 모델 비교 ===


,model,cv_MAE_mean,cv_MAE_std,cv_RMSE_mean,cv_R2_mean
0,Random Forest,8.786343e+05,2.250087e+05,1.217057e+06,-0.529416
1,Baseline - Previous Month,1.161929e+06,3.555479e+05,1.480170e+06,-1.713700
2,Linear Regression,1.702799e+06,1.236690e+06,2.077272e+06,-11.678461


그런데 R²가 왜 이렇게 이상하지?
예를 들어 Linear Regression의 2026-02 R²가 -45.2까지 내려갔지.
이건 지금 한 달을 검증할 때 카테고리 7개밖에 없기 때문이야.

### STEP 20-3. 모델 Freeze

### Train 내부 모델 선택 결과

Train 기간 내부에서 월 단위 Expanding Validation을 수행하였다.

비교한 방법은 다음과 같다.

- Baseline: 직전 달 카테고리 매출을 다음 달 매출로 예측
- Linear Regression
- Random Forest

4개의 검증 월에서 Random Forest의 MAE가 모두 가장 낮았다.

평균 MAE는 대략 다음과 같다.

- Baseline: 약 1,162,000원
- Linear Regression: 약 1,703,000원
- Random Forest: 약 879,000원

Random Forest는 Train 내부 검증에서 Baseline보다 평균 MAE가 약 24% 낮았다.

따라서 Final Test를 확인하기 전에 최종 후보 모델을
Random Forest로 고정한다.

단, 검증 월이 4개이고 각 월의 관측값이 7개 카테고리뿐이므로
현재 결과는 데이터가 충분한 대규모 검증 결과로 해석하지 않는다.

### STEP 20-4. Final Test

In [38]:
# ==================================================
# STEP 20-4. Final Test
# Baseline vs Frozen Random Forest
# ==================================================

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


# --------------------------------------------------
# 1. Feature 구분
# --------------------------------------------------
categorical_features = [
    "category"
]

numeric_features = [
    "target_month_of_year",
    "last_month_sales",
    "recent_3m_avg_sales",
    "last_month_orders",
    "recent_3m_avg_orders",
    "last_month_quantity",
    "recent_3m_avg_quantity",
]


# --------------------------------------------------
# 2. 전처리기
# --------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features,
        ),
    ]
)


# --------------------------------------------------
# 3. Train CV에서 선택한 모델을 그대로 고정
# --------------------------------------------------
frozen_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                random_state=42,
            ),
        ),
    ]
)


# --------------------------------------------------
# 4. Train 전체로 최종 학습
# --------------------------------------------------
frozen_model.fit(
    X_train,
    y_train,
)


# --------------------------------------------------
# 5. Final Test 예측
# --------------------------------------------------

# Baseline:
# 다음 달 매출 = 직전 달 매출
baseline_pred = (
    test_data["last_month_sales"]
    .to_numpy()
)

# Frozen Random Forest
rf_pred = frozen_model.predict(
    X_test
)


# --------------------------------------------------
# 6. 평가 함수
# --------------------------------------------------
def evaluate_model(
    model_name,
    y_true,
    y_pred,
):

    return {
        "model": model_name,

        "test_MAE":
            mean_absolute_error(
                y_true,
                y_pred,
            ),

        "test_RMSE":
            mean_squared_error(
                y_true,
                y_pred,
            ) ** 0.5,

        "test_R2":
            r2_score(
                y_true,
                y_pred,
            ),
    }


# --------------------------------------------------
# 7. Baseline / Random Forest 평가
# --------------------------------------------------
final_results = pd.DataFrame([
    evaluate_model(
        "Baseline - Previous Month",
        y_test,
        baseline_pred,
    ),

    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred,
    ),
])


# --------------------------------------------------
# 8. Baseline 대비 MAE 개선율
# --------------------------------------------------
baseline_mae = final_results.loc[
    final_results["model"]
    .eq("Baseline - Previous Month"),
    "test_MAE",
].iloc[0]

final_results[
    "MAE_improvement_vs_baseline_pct"
] = (
    (
        baseline_mae
        - final_results["test_MAE"]
    )
    / baseline_mae
    * 100
)


# --------------------------------------------------
# 9. 결과 출력
# --------------------------------------------------
print("=== Final Test 기간 ===")
print(
    test_data["target_month"].min(),
    "~",
    test_data["target_month"].max(),
)

print(
    "Final Test 행 수:",
    len(test_data),
)

print("\n=== Final Test 결과 ===")
display(final_results)

=== Final Test 기간 ===
2026-05 ~ 2026-06
Final Test 행 수: 14

=== Final Test 결과 ===


,model,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline - Previous Month,1.051929e+06,1.296306e+06,-2.894701,0.000000
1,Random Forest,1.218784e+06,1.458494e+06,-3.930238,-15.861852


### Final Test 결과

Train 내부 시간순서 검증을 통해 선택한 Random Forest를 고정한 뒤,
아직 모델 선택에 사용하지 않은 2026-05 ~ 2026-06 데이터를 Final Test로 평가하였다.

결과는 다음과 같다.

- Previous Month Baseline
  - MAE: 약 1,052,000원
  - RMSE: 약 1,296,000원
  - R²: 약 -2.895

- Random Forest
  - MAE: 약 1,219,000원
  - RMSE: 약 1,458,000원
  - R²: 약 -3.930
  - Baseline 대비 MAE 개선율: 약 -15.86%

Train 내부 검증에서는 Random Forest가 Baseline보다 좋은 결과를 보였지만,
Final Test에서는 오히려 Baseline보다 MAE가 약 15.86% 높았다.

따라서 현재 Random Forest가 미래 카테고리 매출을 안정적으로 예측한다고 보기 어렵다.

또한 Final Test가 2개월, 14개 관측값으로 매우 작기 때문에
R²를 포함한 평가 결과의 변동성이 클 수 있다는 한계가 있다.

Final Test 결과가 좋지 않더라도 모델을 다시 선택하지 않고,
Train 데이터만으로 선택한 Random Forest의 결과를 그대로 기록한다.

### STEP 20-5. 실제값과 예측값 확인

In [39]:
# ==================================================
# STEP 20-5. 실제값 vs 예측값 및 대표 오차 확인
# ==================================================

prediction_result = test_data[
    [
        "category",
        "target_month",
    ]
].copy()

prediction_result["actual_sales"] = (
    y_test.to_numpy()
)

prediction_result["baseline_prediction"] = (
    baseline_pred
)

prediction_result["rf_prediction"] = (
    rf_pred
)

# Random Forest 잔차
# residual > 0  : 실제값보다 낮게 예측
# residual < 0  : 실제값보다 높게 예측
prediction_result["rf_residual"] = (
    prediction_result["actual_sales"]
    - prediction_result["rf_prediction"]
)

prediction_result["rf_abs_error"] = (
    prediction_result["rf_residual"].abs()
)


print("=== Final Test 실제값 vs 예측값 ===")
display(
    prediction_result.sort_values(
        [
            "target_month",
            "category",
        ]
    )
)


print("\n=== Random Forest 큰 오차 TOP 5 ===")
top_errors = (
    prediction_result
    .sort_values(
        "rf_abs_error",
        ascending=False,
    )
    .head(5)
)

display(top_errors)

=== Final Test 실제값 vs 예측값 ===


,category,target_month,actual_sales,baseline_prediction,rf_prediction,rf_residual,rf_abs_error
6,도서,2026-05,1313000.0,2925000,1969440.0,-656440.0,656440.0
14,뷰티,2026-05,1114000.0,3863000,1586760.0,-472760.0,472760.0
22,생활용품,2026-05,1490000.0,3912000,1661010.0,-171010.0,171010.0
30,스포츠,2026-05,1227000.0,2149000,2336170.0,-1109170.0,1109170.0
38,식품,2026-05,134000.0,1396000,1955505.0,-1821505.0,1821505.0
46,전자기기,2026-05,2008000.0,2039000,2010080.0,-2080.0,2080.0
54,패션,2026-05,777000.0,1269000,1820660.0,-1043660.0,1043660.0
7,도서,2026-06,0.0,1313000,2459985.0,-2459985.0,2459985.0
15,뷰티,2026-06,32000.0,1114000,2077750.0,-2045750.0,2045750.0
23,생활용품,2026-06,963000.0,1490000,2447580.0,-1484580.0,1484580.0



=== Random Forest 큰 오차 TOP 5 ===


,category,target_month,actual_sales,baseline_prediction,rf_prediction,rf_residual,rf_abs_error
31,스포츠,2026-06,386000.0,1227000,3187380.0,-2801380.0,2801380.0
7,도서,2026-06,0.0,1313000,2459985.0,-2459985.0,2459985.0
15,뷰티,2026-06,32000.0,1114000,2077750.0,-2045750.0,2045750.0
38,식품,2026-05,134000.0,1396000,1955505.0,-1821505.0,1821505.0
23,생활용품,2026-06,963000.0,1490000,2447580.0,-1484580.0,1484580.0


In [40]:
# ==================================================
# Final Test 이상 패턴 확인
# 월별 completed 주문 / 매출 확인
# ==================================================

monthly_check = (
    sales_data
    .groupby("month", as_index=False)
    .agg(
        completed_order_count=("order_id", "nunique"),
        total_sales=("sales", "sum"),
        total_quantity=("quantity", "sum"),
    )
)

display(monthly_check)

,month,completed_order_count,total_sales,total_quantity
0,2025-08,18,15621000,127
1,2025-09,13,10190000,102
2,2025-10,26,25766000,248
3,2025-11,12,8812000,93
4,2025-12,14,11501000,111
5,2026-01,22,17423000,158
6,2026-02,17,9749000,100
7,2026-03,14,13429000,130
8,2026-04,23,17553000,181
9,2026-05,11,8063000,93


## Final Test 주요 오차 분석 - 결론에 쓸 수 있는 핵심

Final Test에서 Random Forest는 Baseline보다 낮은 성능을 보였다.

특히 2026년 6월에 큰 과대예측이 여러 카테고리에서 발생하였다.

월별 completed 주문을 추가로 확인한 결과,

- 2026-04: 23건 / 매출 17,553,000원
- 2026-05: 11건 / 매출 8,063,000원
- 2026-06: 4건 / 매출 2,826,000원

으로 Final Test 기간에 주문과 매출이 크게 감소한 것을 확인하였다.

Random Forest는 직전 월뿐 아니라 최근 3개월의 매출, 주문 수, 판매 수량 등의
과거 패턴을 함께 사용했기 때문에 이러한 급격한 감소를 충분히 반영하지 못하고
여러 카테고리의 매출을 과대예측하였다.

반면 Previous Month Baseline은 직전 월 값만 사용하므로
이미 감소한 5월 매출을 6월 예측에 바로 반영하여
Final Test에서는 Random Forest보다 낮은 MAE를 기록하였다.

다만 현재 데이터만으로 2026년 6월의 매출 감소 원인이
실제 사업 환경의 변화인지, 데이터 특성 때문인지는 판단할 수 없다.

따라서 현재 모델은 급격한 판매 수준 변화에 안정적으로 대응한다고 보기 어렵다.

### STEP 21 최종 과제 결론 작성

# Chapter 09 최종 확장 과제 결과

## 1. 새롭게 정의한 예측 문제

과거 카테고리별 판매 데이터를 이용하여
다음 달 카테고리별 completed 주문 매출을 예측하는 회귀 문제를 정의하였다.

예측 단위는 `category × month`이다.

기존 실습의 `order_total`은 주문 상세가 확정되면 정확하게 계산할 수 있었지만,
새로운 문제의 다음 달 매출은 예측 시점에는 아직 알 수 없는 미래의 값이라는 차이가 있다.


## 2. 이 예측이 필요한 이유

다음 달 카테고리별 매출을 미리 알 수 있다면 다음과 같은 의사결정에 활용할 수 있다.

- 재고 계획
- 판매 계획
- 프로모션 검토
- 예산 계획

따라서 단순히 모델을 만들기 위한 예측이 아니라,
미래의 의사결정을 지원할 수 있는 예측 문제로 정의하였다.


## 3. Prediction Contract

- 예측 목적: 다음 달 카테고리별 매출 예측
- Target: `target_next_month_sales`
- 예측 단위: `category × month`
- 예측 시점: 매월 말
- Prediction Horizon: 다음 1개월
- 사용할 정보: 예측 시점까지 확인된 과거 판매 정보
- 실제 활용: 재고, 판매, 프로모션, 예산 계획 지원

예를 들어 2026년 5월 말에는
2026년 5월까지 확인된 데이터만 이용하여
2026년 6월 카테고리별 매출을 예측한다.


## 4. Target과 Feature 설계

Target은 다음 달 카테고리별 completed 주문의 총매출이다.

사용한 Feature는 다음과 같다.

- `category`
- `target_month_of_year`
- `last_month_sales`
- `recent_3m_avg_sales`
- `last_month_orders`
- `recent_3m_avg_orders`
- `last_month_quantity`
- `recent_3m_avg_quantity`

직전 1개월 정보는 가장 최근의 상태를 나타내고,
최근 3개월 평균은 단일 월의 변동을 줄이고 최근 판매 흐름을 표현하기 위해 사용하였다.

원본 데이터 기간은 2025-07-09부터 2026-07-08까지였기 때문에
2025년 7월과 2026년 7월은 완전한 월이 아니었다.

월 전체 매출을 Target으로 사용하기 위해 두 부분 월을 제외하고
2025-08부터 2026-06까지의 완전한 11개월만 사용하였다.

최종적으로 7개 카테고리와 8개의 예측 대상 월을 이용하여
56개의 모델링 행을 구성하였다.


## 5. Leakage Audit 결과

사용한 모든 Feature가 실제 예측 시점에 존재하는 정보인지 확인하였다.

직전 월 매출, 주문 건수, 판매 수량과
최근 3개월 평균값은 모두 예측 대상 월 이전의 데이터만 사용하였다.

`target_month_of_year`는 미래의 결과가 아니라
어느 달을 예측하는지 나타내는 달력 정보이므로 사용할 수 있다고 판단하였다.

반면 `target_next_month_sales`는 다음 달 실제 매출이므로
Feature로 사용할 경우 Target Leakage가 발생한다.

따라서 해당 컬럼은 Feature에서 제외하고 Target으로만 사용하였다.


## 6. 시간 순서 평가 방법

미래 매출을 예측하는 문제이므로 데이터를 무작위로 섞지 않고
시간 순서를 유지하여 Train과 Final Test를 분리하였다.

예측 가능한 월은 총 8개월이었다.

- Train: 2025-11 ~ 2026-04
- Final Test: 2026-05 ~ 2026-06

Train은 42행,
Final Test는 14행이었다.

같은 월의 7개 카테고리가 Train과 Test에 나뉘지 않도록
월 전체를 하나의 시간 단위로 분리하였다.

Train 내부에서도 미래 월을 검증 데이터로 사용하는
월 단위 Expanding Validation을 수행하였다.


## 7. Baseline과 모델 결과

Baseline은 다음과 같이 정의하였다.

`다음 달 매출 = 직전 달 매출`

후보 모델은 다음과 같다.

- Linear Regression
- Random Forest

Train 내부 검증의 평균 MAE는 대략 다음과 같았다.

- Baseline: 약 1,162,000원
- Linear Regression: 약 1,703,000원
- Random Forest: 약 879,000원

Train 내부 검증에서는 Random Forest가 가장 낮은 MAE를 기록하였다.

따라서 Final Test를 확인하기 전에 Random Forest를 최종 후보 모델로 고정하였다.

그러나 Final Test 결과는 다음과 같았다.

- Baseline
  - MAE: 약 1,052,000원
  - RMSE: 약 1,296,000원
  - R²: 약 -2.895

- Random Forest
  - MAE: 약 1,219,000원
  - RMSE: 약 1,458,000원
  - R²: 약 -3.930
  - Baseline 대비 MAE 개선율: 약 -15.86%

즉 Final Test에서는 Random Forest가
Baseline보다 약 15.86% 높은 MAE를 기록하였다.


## 8. MAE / RMSE / R² 해석

Random Forest의 Final Test MAE가 약 1,219,000원이므로
카테고리별 다음 달 매출 예측에서 평균적으로 약 122만 원의 절대오차가 발생하였다.

RMSE는 약 1,458,000원으로 MAE보다 높았다.
이는 일부 관측값에서 상대적으로 큰 오차가 발생했음을 보여준다.

R²는 약 -3.930으로 음수였다.

따라서 현재 Final Test 데이터에서는
Random Forest가 단순한 평균 수준의 예측보다도 안정적인 설명력을 보였다고 보기 어렵다.

단, Final Test가 2개월 × 7개 카테고리인 14개 관측값에 불과하므로
R²의 변동성이 매우 클 수 있다는 한계가 있다.


## 9. 실제값과 예측값 비교 및 주요 오차

Random Forest의 큰 오차는 특히 2026년 6월에 집중되었다.

대표적인 사례는 다음과 같다.

- 스포츠 / 2026-06
  - 실제 매출: 386,000원
  - 예측 매출: 약 3,187,380원
  - 절대오차: 약 2,801,380원

- 도서 / 2026-06
  - 실제 매출: 0원
  - 예측 매출: 약 2,459,985원
  - 절대오차: 약 2,459,985원

- 뷰티 / 2026-06
  - 실제 매출: 32,000원
  - 예측 매출: 약 2,077,750원
  - 절대오차: 약 2,045,750원

월별 completed 주문을 추가로 확인한 결과,

- 2026-04: 23건 / 매출 17,553,000원
- 2026-05: 11건 / 매출 8,063,000원
- 2026-06: 4건 / 매출 2,826,000원

으로 Final Test 기간에 주문과 매출이 크게 감소하였다.

Random Forest는 최근 여러 달의 판매 패턴을 함께 사용했기 때문에
2026년 6월의 급격한 감소를 충분히 반영하지 못하고
여러 카테고리에서 매출을 과대예측하였다.


## 10. 업무적으로 사용할 수 있는가?

현재 Random Forest를 실제 매출 계획에 바로 사용하는 것은 어렵다고 판단하였다.

그 이유는 다음과 같다.

첫째, Final Test에서 Baseline보다 MAE가 약 15.86% 높았다.

둘째, 평균 절대오차가 약 122만 원으로 현재 데이터의 매출 규모를 고려하면
업무 의사결정에 그대로 사용하기에는 오차가 크다.

셋째, 2026년 6월과 같은 급격한 판매 변화에 안정적으로 대응하지 못하였다.

따라서 현재 모델을 단독으로 실제 재고나 예산 결정에 사용하는 것은 적절하지 않다.


## 11. 현재 데이터와 모델의 한계

가장 큰 한계는 데이터 기간이 짧다는 점이다.

완전한 월은 11개월뿐이었고,
최근 3개월 Feature와 다음 달 Target을 구성한 뒤
실제 모델링 가능한 예측 월은 8개월밖에 남지 않았다.

또한 Final Test도 2개월, 14개 관측값으로 매우 작았다.

따라서 특정 월의 급격한 변화가 전체 평가 결과에 큰 영향을 줄 수 있다.

현재 Feature도 과거 매출, 주문 건수, 판매 수량 등
내부 판매 이력에 주로 의존하고 있다.

프로모션, 할인, 휴일, 계절 요인, 재고 상태 등
매출 변화를 설명할 수 있는 다른 정보는 포함되어 있지 않다.

또한 2026년 6월의 매출 감소가 실제 사업 환경의 변화인지,
현재 데이터의 특성 때문인지는 이 데이터만으로 판단할 수 없다.


## 12. 추가 데이터 또는 다음 개선 방향

가장 먼저 더 긴 기간의 데이터를 확보할 필요가 있다.

최소 여러 해의 월별 데이터가 있다면
계절성과 반복적인 패턴을 더 안정적으로 학습할 수 있다.

추가로 다음과 같은 정보를 검토할 수 있다.

- 프로모션 및 할인 여부
- 공휴일과 계절 정보
- 카테고리별 가격 변화
- 재고 및 품절 정보
- 광고 또는 마케팅 활동
- 신규 상품 출시 정보

또한 현재 사용한 최근 3개월 평균이 최적이라고 단정할 수 없으므로,
충분한 데이터가 확보된다면 Train 내부에서
1개월, 3개월, 6개월 등 다양한 과거 기간을 비교할 수 있다.

하지만 현재 Final Test 결과를 보고 Feature나 모델을 다시 선택하면
Test 데이터가 모델 선택에 사용되므로,
현재 Test 결과는 그대로 보존해야 한다.

새로운 데이터와 새로운 미래 기간을 확보한 뒤
개선된 모델을 다시 평가하는 것이 적절하다.

### STEP 22. LLM 활용 내용을 함께 기록합니다

## LLM 활용 기록

### 사용 목적

Chapter 09 확장 과제에서 실제로 의미 있는 회귀 예측 문제를 정의하고,
Prediction Contract, Target/Feature 설계, Leakage Audit,
시간 순서 평가 방법, 모델 결과 해석을 검토하기 위해 ChatGPT를 활용하였다.

단순히 코드를 생성하는 용도가 아니라,
예측 문제 자체가 업무적으로 의미가 있는지 검토하고
분석 과정에서 발생한 의문을 확인하는 데 사용하였다.


### 주요 Prompt 요약

주요 질문과 요청은 다음과 같았다.

- 이미 계산할 수 있는 `order_total`을 왜 예측해야 하는지 질문
- 과거 고객 구매 패턴을 이용한 미래 매출 예측이 의미가 있는지 질문
- 현재 데이터에서 어떤 미래 예측 문제가 적절한지 검토 요청
- 고객별 / 상품별 / 카테고리별 예측 문제의 데이터 적합성 비교
- 다음 달 카테고리별 매출 예측을 위한 Prediction Contract 작성
- `category × month` 형태의 모델링 데이터 생성 코드 작성
- Feature Leakage Audit 검토
- 최근 3개월 평균 Feature를 사용하는 이유 질문
- 전체 기간을 시간순으로 약 80:20으로 분리하는 방법 검토
- Baseline, Linear Regression, Random Forest 비교 코드 작성
- MAE / RMSE / R² 결과 해석
- Final Test에서 큰 오차가 발생한 이유 검토
- 최종 업무 활용 가능성과 데이터 한계 정리


### LLM이 제안한 내용

LLM은 기존 `order_total` 예측보다
미래에 발생하는 값을 Target으로 정의하는 것이 더 의미 있다고 제안하였다.

현재 데이터를 확인한 뒤 다음 세 가지 후보를 비교하였다.

- 고객의 향후 구매금액
- 상품별 다음 기간 판매 수량
- 다음 달 카테고리별 매출

고객별 반복 구매 횟수와 상품별 데이터 희소성을 확인한 결과,
7개의 카테고리로 데이터를 집계하는
`다음 달 카테고리별 completed 주문 매출 예측`을 제안하였다.

또한 다음과 같은 Feature를 제안하였다.

- 직전 월 매출
- 최근 3개월 평균 매출
- 직전 월 주문 건수
- 최근 3개월 평균 주문 건수
- 직전 월 판매 수량
- 최근 3개월 평균 판매 수량
- 카테고리
- 예측 대상 월 정보

평가 방법으로는 시간 순서를 유지한 Train / Final Test 분리,
직전 달 매출을 그대로 사용하는 Baseline,
Linear Regression과 Random Forest 비교를 제안하였다.


### 그대로 사용하지 않고 수정한 부분

LLM의 제안을 그대로 사용하지 않고 여러 부분을 실제 데이터에 맞게 수정하였다.

첫째, 처음에는 전체 13개월을 그대로 월별 데이터로 사용할 수 있다고 생각했지만,
원본 데이터 기간을 확인한 결과 시작일이 2025-07-09,
종료일이 2026-07-08이었다.

따라서 2025년 7월과 2026년 7월은 월 전체 데이터가 아니므로
두 달을 제외하고 2025-08 ~ 2026-06의 완전한 월만 사용하였다.

둘째, 최근 3개월 평균 Feature는 정해진 정답이 아니라
최근 추세를 표현하기 위한 하나의 설계 선택이라는 점을 확인하였다.
현재 데이터가 짧기 때문에 과도하게 긴 기간을 사용하지 않고
직전 1개월과 최근 3개월 정보를 함께 사용하는 방식으로 결정하였다.

셋째, 단순히 전체 56행을 행 단위로 무작위 80:20 분할하지 않았다.
같은 월의 카테고리가 Train과 Test에 섞이지 않도록
예측 대상 월 전체를 시간순으로 분리하였다.

결과적으로

- Train: 2025-11 ~ 2026-04
- Final Test: 2026-05 ~ 2026-06

으로 분리하였다.

넷째, Final Test 결과에서 Random Forest가 Baseline보다 성능이 좋지 않았지만
결과를 보고 다른 모델로 다시 변경하지 않았다.
Train 내부 검증에서 선택한 Random Forest를 그대로 유지하였다.


### 수정 이유

수정의 가장 중요한 기준은
실제 예측 시점과 데이터의 시간 구조를 지키는 것이었다.

부분 월을 포함하면 월 전체 매출이라는 Target 자체가 잘못 정의될 수 있고,
행 단위 무작위 분할을 사용하면 같은 미래 월의 정보가
Train과 Test에 동시에 포함될 수 있기 때문이다.

또한 Final Test를 확인한 뒤 모델을 다시 선택하면
Test 데이터가 모델 선택 과정에 사용되므로
최종 평가의 의미가 사라질 수 있다고 판단하였다.


### 최종 결과를 검증한 방법

먼저 Feature Leakage Audit을 수행하여
모든 사용 Feature가 예측 대상 월 이전에 존재하는 정보인지 확인하였다.

모델링 데이터는 다음과 같이 구성되었다.

- 완전한 월: 11개월
- 카테고리: 7개
- 최종 모델링 행: 56개
- 예측 대상 월: 2025-11 ~ 2026-06

Train 내부에서는 월 단위 Expanding Validation을 사용하여
Baseline, Linear Regression, Random Forest를 비교하였다.

Train 내부 검증에서는 Random Forest의 평균 MAE가 가장 낮아
Final Test 전에 해당 모델을 고정하였다.

그러나 Final Test에서는 다음 결과가 나타났다.

- Baseline MAE: 약 1,052,000원
- Random Forest MAE: 약 1,219,000원
- Random Forest의 Baseline 대비 MAE 개선율: 약 -15.86%

즉 Random Forest는 Final Test에서 Baseline보다 성능이 낮았다.

추가로 실제값과 예측값을 비교하고 큰 오차 사례를 확인하였다.

2026년 6월에는 completed 주문이 4건,
총매출이 2,826,000원으로 크게 감소하였으며,
Random Forest가 이러한 급격한 변화를 충분히 반영하지 못하고
여러 카테고리의 매출을 과대예측한 것을 확인하였다.

최종 판단은 LLM의 설명만 사용하지 않고
실제 데이터, 평가 지표, 월별 주문 및 매출 집계 결과를 직접 확인한 뒤 작성하였다.

## 최종 체크

- [x] Target과 예측 시점을 정의했습니다.
- [x] Leakage Audit을 수행했습니다.
- [x] Target 계산과 관계 검증을 확인했습니다.
- [x] Train과 Final Test가 시간 순서로 엄격히 분리되었습니다.
- [x] 전처리가 Pipeline 내부에서 학습됩니다.
- [x] Train TimeSeriesSplit으로 후보를 비교했습니다.
- [x] Final Test 전에 Selected Model을 고정했습니다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했습니다.
- [x] MAE, RMSE, R²를 함께 해석했습니다.
- [x] 대표 오차 사례를 관찰과 가설로 구분했습니다.
- [x] Validation Evidence가 모두 PASS입니다.
- [x] Internal 결과와 공개 결과를 구분했습니다.
- [x] 낮은 성능도 숨기지 않았습니다.
- [x] 전체 스크립트를 재실행했습니다.
- [x] 최종 Notebook URL을 제출합니다.